# 🔬 OptoBind-AI — Pipeline de Design De Novo de Biosenseurs Optiques

Ce notebook exécute **OptoBind-AI** dans un environnement propre **Python 3.10 natif + PyTorch CUDA** sur un **GPU NVIDIA gratuit dans Google Colab**.

---

### Étape 1 : Vérification du GPU NVIDIA T4
*(Assurez-vous que le GPU est actif dans Exécution > Modifier le type d'exécution > T4 GPU)*

In [ ]:
!nvidia-smi

### Étape 2 : Cloner le projet Optobind-AI

In [ ]:
%cd /content
!rm -rf Optobind-AI
!git clone https://github.com/Lorenzo-colao/Optobind-AI.git
%cd /content/Optobind-AI

!git clone https://github.com/RosettaCommons/RFdiffusion.git 2>/dev/null || true
!git clone https://github.com/dauparas/ProteinMPNN.git 2>/dev/null || true

print("\n✓ Dépôts sources prêts !")

### Étape 3 : Installation de l'environnement Python 3.10 natif

In [ ]:
%%time
# 1. Installer Python 3.10 via apt Ubuntu (rapide et infaillible)
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
!apt-get update -qq > /dev/null
!apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils > /dev/null

# 2. Créer l'environnement virtuel Python 3.10 dédié
!python3.10 -m venv /content/optobind_env

# 3. Installer PyTorch avec CUDA 11.8 officiel pour Python 3.10
!/content/optobind_env/bin/pip install --quiet --upgrade pip
!/content/optobind_env/bin/pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118

# 4. Installer DGL compatible CUDA 11.8 pour Python 3.10
!/content/optobind_env/bin/pip install --quiet dgl -f https://data.dgl.ai/wheels/cu118/repo.html || \
 /content/optobind_env/bin/pip install --quiet dgl -f https://data.dgl.ai/wheels/repo.html || \
 /content/optobind_env/bin/pip install --quiet dgl

# 5. Installer les bibliothèques du pipeline
!/content/optobind_env/bin/pip install --quiet biopython rdkit openpyxl pandas pyyaml tqdm hydra-core omegaconf pynvml decorator e3nn==0.3.3 chai-lab fair-esm
!/content/optobind_env/bin/pip install --quiet git+https://github.com/NVIDIA/dllogger#egg=dllogger

# 6. Installer les sous-modules internes
!/content/optobind_env/bin/bash -c 'cd RFdiffusion/env/SE3Transformer && /content/optobind_env/bin/pip install --quiet .'
!/content/optobind_env/bin/bash -c 'cd RFdiffusion && /content/optobind_env/bin/pip install --quiet --no-deps -e .'

# 7. Télécharger les poids s'ils ne sont pas présents
!mkdir -p weights
!wget -c -P weights/ http://files.ipd.uw.edu/pub/RF-All-Atom/weights/RFDiffusionAA_paper_weights.pt
!wget -c -O weights/ProteinMPNN_v_48_020.pt https://raw.githubusercontent.com/dauparas/ProteinMPNN/main/vanilla_model_weights/v_48_020.pt

print("\n🎉 ENVIRONNEMENT PYTHON 3.10 DÉDIÉ INSTALLÉ AVEC SUCCÈS !")

### Étape 4 : Lancer le pipeline OptoBind-AI sur GPU !

In [ ]:
!/content/optobind_env/bin/python orchestrator.py \
  --target inputs/targets/zn2+_site.pdb \
  --ligand inputs/ligands/DFHBI-1T.sdf

### Étape 5 : Télécharger votre rapport Excel de validation

In [ ]:
import glob
from google.colab import files

# Recherche du rapport généré
reports = glob.glob('outputs/**/reports/report.xlsx', recursive=True)
if reports:
    print(f"Téléchargement de : {reports[-1]}")
    files.download(reports[-1])
else:
    print("Aucun rapport trouvé.")